# REFLECT – Prompt-Following Eval
Benchmarks generator models on rule compliance + LLM-judge quality for single journal entries.

## Config

In [5]:
from pathlib import Path
import re, json
from statistics import mean, pstdev
from typing import Any
import pandas as pd
from llama_index.llms.ollama import Ollama

SCRIPT_DIR       = Path(".").resolve()
EVAL_DATA_PATH   = SCRIPT_DIR / "eval_data"
RESULTS_PATH     = SCRIPT_DIR / "eval_results"
OLLAMA_BASE_URL  = "http://localhost:11434"

# Avoid using the same models as both generator AND judge (self-grading bias).
MODELS_TO_TEST  = ["mistral", "llama3", "gpt-oss:20b"]
JUDGE_MODELS    = ["mistral", "llama3"]

MAX_ENTRIES      = None 
REQUEST_TIMEOUT  = 600.0
OUTPUT_PREFIX    = "prompt_eval"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
print("Config OK")


Config OK


## Prompts

In [6]:
REFLECT_PROMPT = """You are a reflective journaling assistant.
Given the following journal entry, generate a single clarifying question
that helps the writer go deeper on something they mentioned but did not explore.

Rules:
- Ask about ONE specific thing from the entry
- Do not ask yes/no questions
- Do not start with "How does that make you feel"
- Keep it under 20 words

Journal entry:
{journal_text}

Question:"""

JUDGE_PROMPT = """You are evaluating a journaling question generated by an AI assistant.
Respond only with valid JSON, no extra text.

Journal entry:
{journal_text}

Generated question:
{question}

Rate each dimension 1-5:
- specificity: references concrete details from the entry, not generic advice.
- depth: likely to trigger meaningful self-reflection.
- grounding: clearly anchored in this entry rather than a generic journaling question.

{{"specificity": N, "depth": N, "grounding": N, "reason": "one sentence"}}"""
print("Prompts loaded")


Prompts loaded


## Helpers

In [7]:
WORD_RE = re.compile(r"[A-Za-z0-9]+(?:['-][A-Za-z0-9]+)?")
YES_NO_STARTERS = {
    "is","are","am","was","were","do","does","did","can","could",
    "would","will","have","has","had","should","shall","may","might","must",
}
BANNED_OPENER = "how does that make you feel"

def build_llm(model_name: str) -> Ollama:
    extra = {"think": False} if "qwen" in model_name.lower() else {}
    return Ollama(model=model_name, base_url=OLLAMA_BASE_URL,
                  request_timeout=REQUEST_TIMEOUT, additional_kwargs=extra)

def load_test_entries(max_entries):
    files = sorted(EVAL_DATA_PATH.glob("*.txt"))
    entries = [(f.stem, f.read_text(encoding="utf-8", errors="ignore").strip())
               for f in files if f.stat().st_size > 0]
    if max_entries: entries = entries[:max_entries]
    return dict(entries)

def extract_json_dict(raw: str) -> dict:
    text = raw.strip().lstrip("```").lstrip("json").strip().rstrip("```").strip()
    try:
        p = json.loads(text); return p if isinstance(p, dict) else {}
    except json.JSONDecodeError:
        s, e = text.find("{"), text.rfind("}") + 1
        if s == -1 or e <= s: return {}
        try: p = json.loads(text[s:e]); return p if isinstance(p, dict) else {}
        except: return {}

def to_score(v) -> int | None:
    try: s = int(v)
    except: return None
    return max(1, min(5, s))

def analyze_rules(question: str) -> dict:
    clean = " ".join(question.strip().strip('"').split())
    words = WORD_RE.findall(clean)
    wc    = len(words)
    first = words[0].lower() if words else ""
    yn    = first in YES_NO_STARTERS
    ban   = clean.lower().startswith(BANNED_OPENER)
    v     = int(wc > 20) + int(yn) + int(ban)
    s     = 5 - (2 if wc > 20 else 0) - (2 if yn else 0) - (3 if ban else 0)
    return {"word_count": wc, "is_yes_no": yn, "starts_with_banned": ban,
            "strict_rule_pass": v == 0, "rule_following_det": max(1, s)}

def generate_question(llm, journal_text: str) -> str:
    return str(llm.complete(REFLECT_PROMPT.format(journal_text=journal_text))).strip()

def judge_question(judge_llm, journal_text: str, question: str) -> dict:
    raw = str(judge_llm.complete(JUDGE_PROMPT.format(
        journal_text=journal_text, question=question))).strip()
    return extract_json_dict(raw)

def aggregate_judgements(payloads: dict) -> dict:
    metrics = ("specificity", "depth", "grounding")
    per = {}; reasons = []; valid = 0
    for jname, payload in payloads.items():
        row = {m: to_score(payload.get(m)) for m in metrics}
        per[jname] = row
        if any(v is not None for v in row.values()): valid += 1
        r = payload.get("reason","")
        if isinstance(r, str) and r.strip(): reasons.append(f"{jname}: {r.strip()}")
    agg = {}
    for m in metrics:
        vals = [row[m] for row in per.values() if row[m] is not None]
        agg[m] = round(mean(vals), 3) if vals else None
        agg[f"{m}_judge_std"] = (None if not vals else 0.0 if len(vals)==1
                                 else round(pstdev(vals), 3))
    n = max(1, len(payloads))
    agg["judge_valid_count"] = valid
    agg["judge_valid_rate"]  = round(valid / n, 3)
    agg["judge_reasons"]     = " | ".join(reasons)
    agg["judge_scores_json"] = json.dumps(per, ensure_ascii=True)
    return agg

print("Helpers loaded")


Helpers loaded


## Run Eval

In [ ]:
entries = load_test_entries(MAX_ENTRIES)
judge_llms = {j: build_llm(j) for j in JUDGE_MODELS}

print(f"Entries: {len(entries)}  |  Generators: {len(MODELS_TO_TEST)}  |  Judges: {list(judge_llms)}")
rows = []

for model_name in MODELS_TO_TEST:
    print(f"\n── {model_name} ──")
    gen = build_llm(model_name)
    for entry_type, text in entries.items():
        question   = generate_question(gen, text)
        rule_stats = analyze_rules(question)
        payloads   = {j: judge_question(jllm, text, question)
                      for j, jllm in judge_llms.items()}
        agg = aggregate_judgements(payloads)
        print(f"  [{entry_type}] {question}")
        rows.append({"model": model_name, "entry_type": entry_type,
                     "question": question, **rule_stats, **agg})

df = pd.DataFrame(rows)
print("\nDone.")


Entries: 15  |  Generators: 3  |  Judges: ['mistral', 'llama3']

── mistral ──


: 

## Summary

In [ ]:
summary = (
    df.groupby("model")
    .agg(samples=("question","count"),
         specificity=("specificity","mean"), depth=("depth","mean"),
         grounding=("grounding","mean"),
         rule_following_det=("rule_following_det","mean"),
         strict_rule_pass_rate=("strict_rule_pass","mean"),
         judge_valid_rate=("judge_valid_rate","mean"))
    .round(3)
    .sort_values(["specificity","depth","grounding","rule_following_det"], ascending=False)
)

per_path = RESULTS_PATH / f"{OUTPUT_PREFIX}.csv"
sum_path = RESULTS_PATH / f"{OUTPUT_PREFIX}_summary.csv"
df.to_csv(per_path, index=False)
summary.to_csv(sum_path)

print(summary)
print(f"\nSaved: {per_path}\n       {sum_path}")
